# Build Master File
Joins all interim country-month files into a single master panel.

**Output:** `country_month_master.csv` — 1,368 rows × 44 columns  
**Spine:** 19 countries × 72 months (2019-01 → 2024-12)

### Sources joined
| File | Rows | Key notes |
|---|---|---|
| emdat_country_month.csv | 142 | Sparse — event months only |
| noaa_country_month.csv | 77 | Sparse — storm months only |
| unhcr_country_month.csv | 1,320 | Full panel, 19 countries |
| r4v_country_month.csv | 432 | Partial — 6 countries only |
| inform_country_month.csv | 1,296 | Full panel, 18 countries (PRI missing) |
| fts_country_month.csv | 1,368 | Full panel, 19 countries |
| fema_country_month.csv | 72 | PRI only |

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

INTERIM   = Path('/data/interim')   # folder containing all *_country_month.csv files
PROCESSED = Path('/data/processed') # where master will be written
PROCESSED.mkdir(parents=True, exist_ok=True)

## 1. Define the spine
The spine is the authoritative list of every country-month combination we care about.
Every source is left-joined onto it — so missing event months become zeros, not dropped rows.

In [2]:
TIER1 = {
    'BLZ','CRI','CUB','DOM','SLV','GTM','GUY',
    'HTI','HND','JAM','NIC','PAN','PRI','SUR','TTO','BHS','BRB'
}
PRESSURE     = {'COL', 'VEN'}          # displacement origin/transit, not allocation targets
ALL_COUNTRIES = sorted(TIER1 | PRESSURE)
ALL_MONTHS    = pd.date_range('2019-01-01', '2024-12-01', freq='MS')

spine = pd.DataFrame([
    {'iso3': iso3, 'month': month}
    for iso3 in ALL_COUNTRIES
    for month in ALL_MONTHS
])
spine['year']         = spine['month'].dt.year
spine['country_role'] = spine['iso3'].apply(
    lambda x: 'pressure_input' if x in PRESSURE else 'tier1'
)

print(f'Spine: {len(spine)} rows ({len(ALL_COUNTRIES)} countries × {len(ALL_MONTHS)} months)')
spine.head(3)

Spine: 1368 rows (19 countries × 72 months)


,iso3,month,year,country_role
0,BHS,2019-01-01,2019,tier1
1,BHS,2019-02-01,2019,tier1
2,BHS,2019-03-01,2019,tier1


## 2. Load and normalize each source
Each loader standardizes column names and produces a `month` column as datetime.
The key normalization needed:
- `ISO3` (uppercase) → `iso3`
- `year_month` string (e.g. `'2019-08'`) → datetime `month`
- `year` + integer `month` → datetime `month`

In [3]:
# ── EM-DAT ────────────────────────────────────────────────────────────────────
# Sparse: only months with at least one disaster event appear.
# year_month is a string like '2019-08' — append '-01' to parse as datetime.
emdat = pd.read_csv(INTERIM / 'emdat_country_month.csv')
emdat = emdat.rename(columns={'ISO3': 'iso3'})
emdat['month'] = pd.to_datetime(emdat['year_month'] + '-01')
emdat = emdat[[
    'iso3', 'month', 'event_count', 'max_severity', 'mean_severity',
    'sum_affected', 'sum_deaths', 'disaster_types',
    'has_storm', 'has_flood', 'has_earthquake',
    'has_drought', 'has_heatwave', 'compound_event'
]]
print(f'EM-DAT: {len(emdat)} rows')
emdat.head(3)

EM-DAT: 142 rows


,iso3,month,event_count,max_severity,mean_severity,sum_affected,sum_deaths,disaster_types,has_storm,has_flood,has_earthquake,has_drought,has_heatwave,compound_event
0,BHS,2019-09-01,1,0.6876,0.6876,14940.0,356.0,storm,1,0,0,0,0,0
1,BLZ,2020-11-01,1,0.3587,0.3587,60000.0,0.0,storm,1,0,0,0,0,0
2,BLZ,2022-11-01,1,0.3931,0.3931,172150.0,0.0,storm,1,0,0,0,0,0


In [4]:
# ── NOAA IBTrACS ──────────────────────────────────────────────────────────────
# Sparse: only months with storm activity.
# year_month string → datetime, then rename max_wind_score → peak_wind_score
# so the master grid has a consistent column name used by the optimization model.
noaa = pd.read_csv(INTERIM / 'noaa_country_month.csv')
noaa = noaa.rename(columns={'ISO3': 'iso3'})
noaa['month'] = pd.to_datetime(noaa['year_month'] + '-01')
noaa = noaa.rename(columns={
    'max_wind_score':    'peak_wind_score',
    'max_sshs':          'max_sshs_category',
    'had_major_hurricane': 'had_major_hurricane',
})
noaa = noaa[[
    'iso3', 'month', 'storm_count', 'peak_wind_score',
    'max_sshs_category', 'landfall_count', 'storm_names', 'had_major_hurricane'
]]
print(f'NOAA: {len(noaa)} rows')
noaa.head(3)

NOAA: 77 rows


,iso3,month,storm_count,peak_wind_score,max_sshs_category,landfall_count,storm_names,had_major_hurricane
0,BHS,2019-07-01,1,0.1562,-1,1,UNNAMED,0
1,BHS,2019-08-01,1,0.8312,4,0,DORIAN,1
2,BHS,2019-09-01,2,1.0000,5,0,DORIAN|HUMBERTO,1


In [5]:
# ── UNHCR ─────────────────────────────────────────────────────────────────────
# Full panel (19 countries × 72 months = 1,320 rows, PRI absent).
# month column is an integer (1-12) — combine with year to get datetime.
# Rename to avoid collision with R4V displacement_source column.
unhcr = pd.read_csv(INTERIM / 'unhcr_country_month.csv')
unhcr = unhcr.rename(columns={'ISO3': 'iso3'})
unhcr['month'] = pd.to_datetime(
    unhcr['year'].astype(str) + '-' +
    unhcr['month'].astype(str).str.zfill(2) + '-01'
)
unhcr = unhcr[[
    'iso3', 'month',
    'monthly_displaced_in_baseline',
    'monthly_displaced_out_baseline',
    'monthly_pressure_baseline',
    'displacement_source'
]].rename(columns={
    'monthly_displaced_in_baseline':  'unhcr_displaced_in',
    'monthly_displaced_out_baseline': 'unhcr_displaced_out',
    'monthly_pressure_baseline':      'unhcr_pressure',
    'displacement_source':            'unhcr_source',
})
print(f'UNHCR: {len(unhcr)} rows')
unhcr.head(3)

UNHCR: 1320 rows


,iso3,month,unhcr_displaced_in,unhcr_displaced_out,unhcr_pressure,unhcr_source
0,BHS,2019-01-01,1.92,85.75,87.67,unhcr_annual_proxy
1,BHS,2019-02-01,1.92,85.75,87.67,unhcr_annual_proxy
2,BHS,2019-03-01,1.92,85.75,87.67,unhcr_annual_proxy


In [6]:
# ── R4V ───────────────────────────────────────────────────────────────────────
# Partial panel: COL, CRI, DOM, GUY, PAN, TTO only (Venezuelan displacement).
# Where R4V exists it overrides UNHCR as the displacement source.
r4v = pd.read_csv(INTERIM / 'r4v_country_month.csv')
r4v['month'] = pd.to_datetime(r4v['month'])
r4v = r4v[['iso3', 'month', 'venezuelan_displaced', 'displacement_source']].rename(
    columns={'displacement_source': 'r4v_source'}
)
print(f'R4V: {len(r4v)} rows | countries: {sorted(r4v["iso3"].unique())}')
r4v.head(3)

R4V: 432 rows | countries: ['COL', 'CRI', 'DOM', 'GUY', 'PAN', 'TTO']


,iso3,month,venezuelan_displaced,r4v_source
0,COL,2019-01-01,374105,r4v_backfill_dec2020_proj
1,COL,2019-02-01,374105,r4v_backfill_dec2020_proj
2,COL,2019-03-01,374105,r4v_backfill_dec2020_proj


In [7]:
# ── INFORM ────────────────────────────────────────────────────────────────────
# Full panel for 18 countries (PRI absent from INFORM entirely).
# Annual scores forward-filled to monthly — no within-year variation by design.
inform = pd.read_csv(INTERIM / 'inform_country_month.csv')
inform['month'] = pd.to_datetime(inform['month'])
inform = inform[[
    'iso3', 'month', 'inform_risk_score', 'hazard_exposure',
    'vulnerability', 'lack_coping_capacity', 'natural_hazard',
    'socioeconomic_vulnerability', 'infrastructure_coping'
]]
print(f'INFORM: {len(inform)} rows | PRI absent: {"PRI" not in inform["iso3"].unique()}')
inform.head(3)

INFORM: 1296 rows | PRI absent: True


,iso3,month,inform_risk_score,hazard_exposure,vulnerability,lack_coping_capacity,natural_hazard,socioeconomic_vulnerability,infrastructure_coping
0,BHS,2019-01-01,2.2,1.9,1.7,3.1,3.5,2.0,2.4
1,BHS,2019-02-01,2.2,1.9,1.7,3.1,3.5,2.0,2.4
2,BHS,2019-03-01,2.2,1.9,1.7,3.1,3.5,2.0,2.4


In [8]:
# ── OCHA FTS ──────────────────────────────────────────────────────────────────
# Sparse: only country-months with paid flows appear (603/1224 tier1 rows).
# Missing months are zero-filled at join. flow_count = distinct flows after dedup.
# PRI: effectively unfunded via FTS ($19,800 total, 2019 only) — FEMA proxy carries it.
ocha = pd.read_csv(INTERIM / 'fts_flows_country_month.csv')
ocha['month'] = pd.to_datetime(
    ocha['year'].astype(str) + '-' +
    ocha['month'].astype(str).str.zfill(2) + '-01'
)
ocha = ocha[['iso3', 'month', 'funding_usd', 'flow_count', 'sector_mix', 'has_cerf', 'has_echo']]
print(f'OCHA FTS: {len(ocha)} rows (sparse — zero-filled at join)')
ocha.head(3)

OCHA FTS: 742 rows (sparse — zero-filled at join)


,iso3,month,funding_usd,flow_count,sector_mix,has_cerf,has_echo
0,BHS,2019-03-01,19800,1,Education,0,0
1,BHS,2019-09-01,1202150,2,Protection | Unearmarked,1,0
2,BHS,2019-10-01,1233277,5,Coordination and support services | Emergency ...,1,0


In [9]:
# ── FEMA ─────────────────────────────────────────────────────────────────────
# PRI only (72 rows). Provides disaster signal for Puerto Rico
# in place of UNHCR/R4V which don't cover US territories.
fema = pd.read_csv(INTERIM / 'fema_country_month.csv')
fema['month'] = pd.to_datetime(fema['month'])
fema = fema[[
    'iso3', 'month', 'disaster_count', 'pa_declared',
    'ia_declared', 'major_dr_declared', 'incident_types', 'fema_event_flag'
]]
print(f'FEMA: {len(fema)} rows | countries: {fema["iso3"].unique()}')
fema.head(3)

FEMA: 72 rows | countries: ['PRI']


,iso3,month,disaster_count,pa_declared,ia_declared,major_dr_declared,incident_types,fema_event_flag
0,PRI,2019-01-01,0,0,0,0,NaN,fema_no_event
1,PRI,2019-02-01,0,0,0,0,NaN,fema_no_event
2,PRI,2019-03-01,0,0,0,0,NaN,fema_no_event


## 3. Join everything to the spine
All joins are **left joins on `[iso3, month]`** so the spine drives the row count.
Sparse sources (EM-DAT, NOAA) produce NaN for non-event months — filled with 0 below.

In [10]:
master = spine.copy()
master = master.merge(emdat,  on=['iso3', 'month'], how='left')
master = master.merge(noaa,   on=['iso3', 'month'], how='left')
master = master.merge(unhcr,  on=['iso3', 'month'], how='left')
master = master.merge(r4v,    on=['iso3', 'month'], how='left')
master = master.merge(inform, on=['iso3', 'month'], how='left')
master = master.merge(ocha,   on=['iso3', 'month'], how='left')
master = master.merge(fema,   on=['iso3', 'month'], how='left')

print(f'After joins: {len(master)} rows × {len(master.columns)} cols')

After joins: 1368 rows × 46 cols


## 4. Fill nulls
Three categories:
- **Event columns** (EM-DAT, NOAA): NaN means no event → fill with `0`
- **String columns**: NaN means no event → fill with `''`  
- **Structural NaN** (PRI UNHCR/INFORM, VEN R4V): leave as NaN — these are genuinely absent

In [11]:
# Numeric event columns → 0
zero_fill_cols = [
    'event_count', 'max_severity', 'mean_severity', 'sum_affected', 'sum_deaths',
    'has_storm', 'has_flood', 'has_earthquake', 'has_drought', 'has_heatwave',
    'compound_event',
    'storm_count', 'peak_wind_score', 'max_sshs_category',
    'landfall_count', 'had_major_hurricane',
    'disaster_count', 'pa_declared', 'ia_declared', 'major_dr_declared',
    'funding_usd', 'flow_count', 'has_cerf', 'has_echo',
]
for col in zero_fill_cols:
    if col in master.columns:
        master[col] = master[col].fillna(0)

# Integer types for flag columns
for col in ['flow_count', 'has_cerf', 'has_echo',
            'disaster_count', 'pa_declared', 'ia_declared', 'major_dr_declared',
            'had_major_hurricane']:
    if col in master.columns:
        master[col] = master[col].astype(int)

# String event columns → empty string
for col in ['disaster_types', 'storm_names', 'incident_types', 'sector_mix']:
    if col in master.columns:
        master[col] = master[col].fillna('')

# Flag columns → descriptive string
master['fema_event_flag'] = master['fema_event_flag'].fillna('not_applicable')

## 5. Consolidate displacement
R4V is the preferred displacement source for the 6 countries it covers (COL, CRI, DOM, GUY, PAN, TTO) because it specifically tracks Venezuelan refugees using population projections.

For all other countries, fall back to the UNHCR monthly proxy.

In [12]:
# displaced_monthly: R4V where available, UNHCR elsewhere
master['displaced_monthly'] = master['venezuelan_displaced'].where(
    master['venezuelan_displaced'].notna(),
    master['unhcr_displaced_in']
)

# displacement_source: which source was actually used
master['displacement_source'] = master['r4v_source'].where(
    master['r4v_source'].notna(),
    master['unhcr_source']
)

print('Displacement source breakdown:')
print(master['displacement_source'].value_counts().to_string())

Displacement source breakdown:
displacement_source
unhcr_annual_proxy           888
r4v                          276
r4v_backfill_dec2020_proj     72
r4v_dec2020_proj              72
r4v_from_2023-24_combined     12


## 6. QA — understand every remaining null
After the fills above, every remaining NaN should be explainable.

In [13]:
null_counts = master.isnull().sum()
null_counts = null_counts[null_counts > 0]
print('Remaining nulls:')
print(null_counts.to_string())
print()
print('All expected:')
print('  unhcr_* (48):             PRI is a US territory — UNHCR does not track it')
print('  inform_* (72):            PRI is not in the INFORM database')
print('  displaced_monthly (48):   PRI has no UNHCR or R4V coverage (FEMA used instead)')
print('  venezuelan_displaced (936): 13 countries not in R4V — uses UNHCR proxy')

Remaining nulls:
unhcr_displaced_in              48
unhcr_displaced_out             48
unhcr_pressure                  48
unhcr_source                    48
venezuelan_displaced           936
r4v_source                     936
inform_risk_score               72
hazard_exposure                 72
vulnerability                   72
lack_coping_capacity            72
natural_hazard                  72
socioeconomic_vulnerability     72
infrastructure_coping           72
displaced_monthly               48
displacement_source             48

All expected:
  unhcr_* (48):             PRI is a US territory — UNHCR does not track it
  inform_* (72):            PRI is not in the INFORM database
  displaced_monthly (48):   PRI has no UNHCR or R4V coverage (FEMA used instead)
  venezuelan_displaced (936): 13 countries not in R4V — uses UNHCR proxy


In [14]:
# Spot check — Haiti Aug 2021 (earthquake month)
# Expect: event_count > 0, high severity, high funding, no storm activity
master[
    (master['iso3'] == 'HTI') & (master['month'] == '2021-08-01')
][[
    'iso3', 'month', 'event_count', 'max_severity',
    'inform_risk_score', 'funding_usd', 'displaced_monthly'
]]

,iso3,month,event_count,max_severity,inform_risk_score,funding_usd,displaced_monthly
751,HTI,2021-08-01,2.0,0.939,6.3,26339002.0,0.0


In [15]:
master[
    (master['iso3'] == 'BHS') & (master['month'] == '2019-09-01')
][[
    'iso3', 'month', 'storm_count', 'peak_wind_score', 'max_sshs_category', 'storm_names'
]]

,iso3,month,storm_count,peak_wind_score,max_sshs_category,storm_names
8,BHS,2019-09-01,2.0,1.0,5.0,DORIAN|HUMBERTO


## 7. Write master grid

In [16]:
master = master.sort_values(['iso3', 'month']).reset_index(drop=True)
master.to_csv(PROCESSED / 'country_month_master.csv', index=False)

print(f'Wrote country_month_master.csv')
print(f'  Rows    : {len(master)}')
print(f'  Columns : {len(master.columns)}')
print(f'  Countries: {sorted(master["iso3"].unique())}')
print()
print('Columns in master:')
for c in master.columns:
    print(f'  {c}')

Wrote country_month_master.csv
  Rows    : 1368
  Columns : 48
  Countries: ['BHS', 'BLZ', 'BRB', 'COL', 'CRI', 'CUB', 'DOM', 'GTM', 'GUY', 'HND', 'HTI', 'JAM', 'NIC', 'PAN', 'PRI', 'SLV', 'SUR', 'TTO', 'VEN']

Columns in master:
  iso3
  month
  year
  country_role
  event_count
  max_severity
  mean_severity
  sum_affected
  sum_deaths
  disaster_types
  has_storm
  has_flood
  has_earthquake
  has_drought
  has_heatwave
  compound_event
  storm_count
  peak_wind_score
  max_sshs_category
  landfall_count
  storm_names
  had_major_hurricane
  unhcr_displaced_in
  unhcr_displaced_out
  unhcr_pressure
  unhcr_source
  venezuelan_displaced
  r4v_source
  inform_risk_score
  hazard_exposure
  vulnerability
  lack_coping_capacity
  natural_hazard
  socioeconomic_vulnerability
  infrastructure_coping
  funding_usd
  flow_count
  sector_mix
  has_cerf
  has_echo
  disaster_count
  pa_declared
  ia_declared
  major_dr_declared
  incident_types
  fema_event_flag
  displaced_monthly
  displace